train.py
--------
Trains an XGBoost classifier on the processed appointments data.
Handles class imbalance with SMOTE, tunes with GridSearchCV,
evaluates with multiple metrics, and saves the model + scaler.

Run:  python train.py

In [7]:
!jupyter nbconvert --to script preprocess.ipynb

[NbConvertApp] Converting notebook preprocess.ipynb to script
[NbConvertApp] Writing 4424 bytes to preprocess.py


In [1]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'data', 'model', 'plots', 'preprocess.ipynb', 'preprocess.py', 'processed_appointments.csv', 'raw_appointments.csv', 'requirements.txt', 'train.ipynb', '__pycache__']


In [2]:
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection   import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing     import StandardScaler
from sklearn.metrics           import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve
)
from imblearn.over_sampling    import SMOTE
from xgboost                   import XGBClassifier

In [3]:
from preprocess import load_data, clean, engineer_features, encode_and_select

In [5]:
df = load_data()
df = clean(df)
df = engineer_features(df)
X, y = encode_and_select(df)

Loaded 110,527 rows × 14 columns
After cleaning: 110,521 rows
Features engineered. Total columns: 23


In [6]:
# CONFIG
RANDOM_STATE  = 42
TEST_SIZE     = 0.20
MODEL_DIR     = "model"
PLOTS_DIR     = "plots"
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(PLOTS_DIR,  exist_ok=True)

In [7]:
# 1. LOAD & PREPARE DATA
def prepare_data():
    print("=" * 55)
    print("  PATIENT NO-SHOW PREDICTION — TRAINING PIPELINE")
    print("=" * 55)

    df = load_data()
    df = clean(df)
    df = engineer_features(df)
    X, y = encode_and_select(df)

    # Handle any remaining NaNs
    X = X.fillna(X.median(numeric_only=True))

    print(f"\nClass distribution before SMOTE:")
    print(f"  Show up  (0): {(y==0).sum():,}  ({(y==0).mean()*100:.1f}%)")
    print(f"  No-show  (1): {(y==1).sum():,}  ({(y==1).mean()*100:.1f}%)")

    return X, y

In [8]:
# 2. SPLIT + SCALE
def split_scale(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    print(f"\nTrain size : {X_train_sc.shape[0]:,}")
    print(f"Test  size : {X_test_sc.shape[0]:,}")

    return X_train_sc, X_test_sc, y_train, y_test, scaler, X_train.columns.tolist()

In [9]:
# 3. SMOTE — balance training set
def apply_smote(X_train, y_train):
    sm = SMOTE(random_state=RANDOM_STATE)
    X_res, y_res = sm.fit_resample(X_train, y_train)

    print(f"\nAfter SMOTE:")
    print(f"  Show up  (0): {(y_res==0).sum():,}")
    print(f"  No-show  (1): {(y_res==1).sum():,}")

    return X_res, y_res

In [10]:
# 4. HYPERPARAMETER TUNING
def tune_model(X_train, y_train):
    print("\nRunning GridSearchCV (this may take a few minutes)...")

    param_grid = {
        "n_estimators":     [100, 200],
        "max_depth":        [3, 5],
        "learning_rate":    [0.05, 0.1],
        "subsample":        [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }

    xgb = XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    grid = GridSearchCV(
        xgb, param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X_train, y_train)

    print(f"\nBest CV ROC-AUC : {grid.best_score_:.4f}")
    print(f"Best params     : {grid.best_params_}")

    return grid.best_estimator_, grid.best_params_

In [21]:
import os

MODEL_DIR = "model"
PLOTS_DIR = "plots"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

In [23]:
def evaluate(model, X_test, y_test, feature_names):
    y_pred = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)

    print("\n" + "=" * 40)
    print("  TEST SET RESULTS")
    print("=" * 40)
    print(f"  Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  ROC-AUC   : {roc_auc:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Show up", "No-show"]
        )
    )
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Show up", "No-show"],
        yticklabels=["Show up", "No-show"],
        ax=axes[0]
    )

    axes[0].set_title("Confusion Matrix")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)

    axes[1].plot(
        fpr,
        tpr,
        lw=2,
        label=f"AUC = {roc_auc:.3f}"
    )

    axes[1].plot([0, 1], [0, 1], linestyle="--")

    axes[1].set_title("ROC Curve")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/evaluation.png", dpi=150)
    plt.close()

    print(f"\nEvaluation plot saved → {PLOTS_DIR}/evaluation.png")

    # Feature Importance
    importances = model.feature_importances_

    feat_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })

    feat_df = feat_df.sort_values(
        by="importance",
        ascending=False
    )

    plt.figure(figsize=(10, 7))

    plt.barh(
        feat_df["feature"][::-1],
        feat_df["importance"][::-1]
    )

    plt.title("Feature Importance (XGBoost)")
    plt.xlabel("Importance Score")

    plt.tight_layout()
    plt.savefig(
        f"{PLOTS_DIR}/feature_importance.png",
        dpi=150
    )
    plt.close()

    print(
        f"Feature importance plot saved → "
        f"{PLOTS_DIR}/feature_importance.png"
    )

    return acc, roc_auc, feat_df

In [24]:
# 6. SAVE MODEL
def save_artifacts(model, scaler, feature_names, best_params, acc, roc_auc):
    artifacts = {
        "model":         model,
        "scaler":        scaler,
        "feature_names": feature_names,
        "best_params":   best_params,
        "metrics": {
            "accuracy": round(acc, 4),
            "roc_auc":  round(roc_auc, 4),
        }
    }
    path = f"{MODEL_DIR}/noshow_model.pkl"
    with open(path, "wb") as f:
        pickle.dump(artifacts, f)
    print(f"\nModel artifacts saved → {path}")

In [25]:
# RUN COMPLETE PIPELINE
X, y = prepare_data()

# Split and scale
X_train, X_test, y_train, y_test, scaler, cols = split_scale(X, y)

# Skip SMOTE due to threadpool/OpenBLAS issue
X_train_res = X_train
y_train_res = y_train

print("\nTraining XGBoost model...")

model, best_params = tune_model(X_train_res, y_train_res)

acc, roc_auc, feat_df = evaluate(
    model,
    X_test,
    y_test,
    cols
)

save_artifacts(
    model,
    scaler,
    cols,
    best_params,
    acc,
    roc_auc
)

print("\n TRAINING COMPLETE")
print(f"Accuracy : {acc*100:.2f}%")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nTop 5 Most Important Features:")
print(feat_df.head(5).to_string(index=False))

  PATIENT NO-SHOW PREDICTION — TRAINING PIPELINE
Loaded 110,527 rows × 14 columns
After cleaning: 110,521 rows
Features engineered. Total columns: 23

Class distribution before SMOTE:
  Show up  (0): 88,207  (79.8%)
  No-show  (1): 22,314  (20.2%)

Train size : 88,416
Test  size : 22,105

Training XGBoost model...

Running GridSearchCV (this may take a few minutes)...
Fitting 3 folds for each of 32 candidates, totalling 96 fits

Best CV ROC-AUC : 0.7514
Best params     : {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}

  TEST SET RESULTS
  Accuracy  : 0.8017 (80.17%)
  ROC-AUC   : 0.7454

Classification Report:
              precision    recall  f1-score   support

     Show up       0.81      0.99      0.89     17642
     No-show       0.58      0.06      0.11      4463

    accuracy                           0.80     22105
   macro avg       0.69      0.53      0.50     22105
weighted avg       0.76      0.80      0.73     22105


In [26]:
import os

print(os.path.exists("model/noshow_model.pkl"))

True
